In [1]:
import os

In [2]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change directory:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

## 1. Update the config.yaml 

In [5]:
# model_trainer:
#   root_dir: artifacts/model_trainer  # Directory where the trained model and outputs will be saved
#   data_path: artifacts/data_transformation/samsumdata/samsum_dataset # Path to the transformed dataset used for training
#   model_ckpt: google/pegasus-cnn_dailymail # Pretrained PEGASUS model checkpoint loaded from Hugging Face
#   model_ckpt: t5-small # Pretrained T5 model checkpoint loaded from Hugging Face

## 2. Update the params.py

In [6]:
# TrainingArguments:
#   num_train_epochs: 1
#   warmup_steps: 500
#   per_device_train_batch_size: 1
#   weight_decay: 0.01
#   logging_steps: 10
#   evaluation_strategy: steps
#   eval_steps: 500
#   save_steps: 1e6
#   gradient_accumulation_steps: 16

## 3. Update/Create the Entity:

In [7]:
from dataclasses import dataclass   # used to create simple config classes without writing boilerplate code
from pathlib import Path            # used to handle file and folder paths in a clean way


@dataclass(frozen=True)            # makes this class immutable (values cannot be changed after creation)
class ModelTrainerConfig:

    root_dir: Path                  # folder where trained model and outputs will be saved
    data_path: Path                # path to processed dataset used for training
    model_ckpt: Path              # pretrained model checkpoint (base model to fine-tune)

    num_train_epochs: int          # number of times the model sees the entire dataset
    warmup_steps: int             # steps to gradually increase learning rate at start of training
    per_device_train_batch_size: int  # number of samples processed per GPU/CPU at once
    weight_decay: float           # regularization to reduce overfitting
    logging_steps: int            # how often to log training progress
    evaluation_strategy: str      # when to run evaluation (e.g., 'steps' or 'epoch')
    eval_steps: int               # how often evaluation runs during training
    save_steps: float             # how often to save model checkpoints during training
    gradient_accumulation_steps: int  # number of steps to accumulate gradients before updating model

## 4. Update the Configuration Manager:

In [8]:
from textsummarizer.constants import *  # imports file paths and constant values used across the project
from textsummarizer.utils.common import read_yaml, create_directories  # helper functions to read YAML config and create folders


class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,   # path to config.yaml (project settings)
        params_filepath = PARAMS_FILE_PATH):  # path to params.yaml (training hyperparameters)

        self.config = read_yaml(config_filepath)  # loads configuration file into Python object
        self.params = read_yaml(params_filepath)   # loads parameters (like epochs, batch size, etc.)

        create_directories([self.config.artifacts_root])  # creates main output folder for storing results


    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config.model_trainer  # gets model training settings from config.yaml
        params = self.params.TrainingArguments  # gets training hyperparameters from params.yaml

        create_directories([config.root_dir])  # creates folder to save trained model

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,   # where trained model will be saved
            data_path=config.data_path, # path to training dataset
            model_ckpt=config.model_ckpt, # pretrained model to fine-tune (e.g., Pegasus)

            num_train_epochs=params.num_train_epochs,  # number of training cycles
            warmup_steps=params.warmup_steps,          # steps for gradual learning start
            per_device_train_batch_size=params.per_device_train_batch_size,  # batch size per device
            weight_decay=params.weight_decay,          # prevents overfitting
            logging_steps=params.logging_steps,        # how often logs are printed
            evaluation_strategy=params.evaluation_strategy,  # when to evaluate model
            eval_steps=params.eval_steps,     # likely bug: should be eval_steps not evaluation_strategy
            save_steps=params.save_steps,              # how often model is saved
            gradient_accumulation_steps=params.gradient_accumulation_steps  # simulates larger batch size
        )

        return model_trainer_config  # returns all training settings in one object

## 5. Update the Components:

In [9]:
from transformers import TrainingArguments, Trainer  # tools to define and run training process (Trainer = training engine)
from transformers import DataCollatorForSeq2Seq  # prepares batches for seq2seq or text-to-text tasks like summarization
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer  # loads pretrained model + tokenizer automatically
from datasets import load_dataset, load_from_disk  # loads dataset from HuggingFace or local disk. loads dataset saved on your local machine
import torch  # used to check GPU availability and run model on GPU/CPU. used for GPU/CPU selection and tensor operations
import os  # used for saving files and creating paths

c:\Users\paric\Documents\NLP\Text-Summarizer\texts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# defines a class to handle model training
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig): # receives training configuration (all settings from config files)
        self.config = config  # stores training configuration. stores config inside the class so other functions can use it

    # main function that runs training process
    def train(self):

        # use GPU if available else CPU
        device = "cuda" if torch.cuda.is_available() else "cpu" # checks if GPU exists → uses GPU else CPU

        # load tokenizer and model from pretrained checkpoint
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt) # loads tokenizer (converts text ↔ numbers) from pretrained model
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device) # loads Pegasus model and sends it to GPU/CPU

        # prepares batches for seq2seq training (handles padding + labels)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus) # makes batches properly formatted for training (handles padding+ labels for training, etc.)

        # load processed dataset from disk
        dataset_samsum_pt = load_from_disk(self.config.data_path) # # loads preprocessed dataset from local folder or disk (train + validation)

        # =========================================================================
        # OPTION 1: CONFIG-BASED (BEST PRACTICE - RECOMMENDED) - USING yaml FILE
        # ==========================================================================
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,  # where model and checkpoints will be saved
            num_train_epochs=self.config.num_train_epochs,  # training epochs. number of full passes over dataset
            warmup_steps=self.config.warmup_steps,  # gradual warm-up for stable training
            per_device_train_batch_size=self.config.per_device_train_batch_size,  # training batch size per device
            per_device_eval_batch_size=self.config.per_device_train_batch_size,  # evaluation batch size
            weight_decay=self.config.weight_decay,  # helps reduce overfitting
            logging_steps=self.config.logging_steps,  # logging frequency (how often logs are shown)
            #evaluation_strategy=self.config.evaluation_strategy,
            eval_strategy=self.config.evaluation_strategy,  # when to evaluate (steps/epoch)
            # evaluation_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,  # evaluation interval. how often evaluation runs
            save_steps=1e6,  # save model at end. how often model is saved(very large = save once at end)
            # fp16=True, # Use 16-bit floating point precision instead of normal 32-bit
            gradient_accumulation_steps=self.config.gradient_accumulation_steps  # simulate larger batch size
        )
        # ==========================================================
        # OPTION 2: HARDCODED (ONLY FOR TESTING)
        # ==========================================================
        """
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,  # where model will be saved
            num_train_epochs=1,  # how many times model sees data. model sees dataset only once
            warmup_steps=500,  # slow start for stable training
            per_device_train_batch_size=1,  # training batch size. very small batch (good for weak GPU)
            per_device_eval_batch_size=1,  # evaluation batch size
            weight_decay=0.01,  # prevents overfitting. standard overfitting control
            logging_steps=10,  # show logs every 10 steps
            evaluation_strategy='steps',  # evaluate during training
            eval_steps=500,  # evaluate every 500 steps
            save_steps=1e6,  # save model rarely (almost at end)
            gradient_accumulation_steps=16  # simulate bigger batch using memory efficiency
        )
        """


        # create Trainer object. 
        trainer = Trainer(  # creates training engine
            model=model_pegasus,  # model to train
            args=trainer_args,  # training configuration or setting
            #tokenizer=tokenizer,  # tokenizer for text processing
            data_collator=seq2seq_data_collator,  # batch formatting (prepares batches)
            train_dataset=dataset_samsum_pt["test"],  # training data (I use test data for get faster result)
            eval_dataset=dataset_samsum_pt["validation"]  # validation data
        )

        # start training process
        trainer.train()

        # save trained model
        model_pegasus.save_pretrained(
            os.path.join(self.config.root_dir, "pegasus-samsum-model")
        )

        # save tokenizer for later use
        tokenizer.save_pretrained(
            os.path.join(self.config.root_dir, "tokenizer")
        )

## 6. Update the Pipelines:

In [ ]:
try:
    config = ConfigurationManager() # loads configuration (config.yaml + params.yaml)
    model_trainer_config = config.get_model_trainer_config()     # prepares all training settings (data path, model, hyperparameters)
    model_trainer_config = ModelTrainer(config=model_trainer_config)  # creates ModelTrainer object using prepared config
    model_trainer_config.train()     # starts training the model
except Exception as e: # stops execution and shows error if anything fails
    raise e

[2026-05-13 14:28:16,455: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-13 14:28:16,464: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-13 14:28:16,466: INFO: common: created directory at: artifacts]
[2026-05-13 14:28:16,467: INFO: common: created directory at: artifacts/model_trainer]
[2026-05-13 14:28:16,645: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-05-13 14:28:16,748: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-13 14:28:16,853: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 14:28:16,940: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTT

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3492.08it/s]

[2026-05-13 14:28:17,844: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]


[2026-05-13 14:28:17,844: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]


c:\Users\paric\Documents\NLP\Text-Summarizer\texts\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


IndexError: index out of range in self